In [20]:
# ============================================================
# Official PaSST using hear21passt - Full One-Cell Code
# YMIR / YMED Dataset
# Split-first protocol to avoid data leakage
# ============================================================

import os
import csv
import sys
import subprocess
import importlib.util
import numpy as np
import pandas as pd
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# Install / Import Required Packages
# ============================================================

if importlib.util.find_spec("torch") is None:
    print("Installing torch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchaudio", "-q"])

if importlib.util.find_spec("hear21passt") is None:
    print("Installing hear21passt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hear21passt", "-q"])

import torch
from hear21passt.base import get_basic_model


# ============================================================
# Dataset Settings
# ============================================================

DATA_PATH = "/root/YMED/"
GENRES = ["adeni", "hadrami", "lahji", "sanaai", "tihami"]

SR = 32000                  # PaSST expects 32 kHz
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = 6.0

TEST_SIZE = 0.2
RANDOM_STATE = 42

SAVE_DIR = "/root/autodl-tmp/official_passt_results/"
EMB_DIR = os.path.join(SAVE_DIR, "embeddings")
META_DIR = os.path.join(SAVE_DIR, "metadata")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)


# ============================================================
# Official PaSST Model
# ============================================================

def create_official_passt_model(device=None):
    """
    Official pretrained PaSST using hear21passt.

    Input:
        Raw waveform at 32000 Hz.

    Output:
        AudioSet logits / pretrained representation.
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    passt_model = get_basic_model(mode="logits")
    passt_model.to(device)
    passt_model.eval()

    print("=" * 70)
    print("Official pretrained PaSST loaded successfully.")
    print("Interface: hear21passt.base.get_basic_model(mode='logits')")
    print("Device:", device)
    print("Expected input: raw waveform at 32000 Hz.")
    print("=" * 70)

    return passt_model, device


# ============================================================
# Collect Original Files
# ============================================================

def collect_original_files(data_path, genres):
    """
    Collect original WAV files before segmentation.
    This prevents data leakage.
    """

    file_paths = []
    labels = []

    for idx, genre in enumerate(genres):
        genre_dir = os.path.join(data_path, genre)

        if not os.path.isdir(genre_dir):
            raise FileNotFoundError(f"Genre folder not found: {genre_dir}")

        wav_files = sorted([
            f for f in os.listdir(genre_dir)
            if f.lower().endswith(".wav")
        ])

        print(f"{genre}: {len(wav_files)} original WAV files")

        for filename in wav_files:
            file_paths.append(os.path.join(genre_dir, filename))
            labels.append(idx)

    return np.array(file_paths), np.array(labels)


# ============================================================
# Audio Loading and Segmentation
# ============================================================

def load_audio_30s(file_path, sr=SR, duration=FILE_DURATION):
    """
    Load audio at 32 kHz and force it to exactly 30 seconds.
    """

    target_len = int(sr * duration)

    audio, _ = librosa.load(file_path, sr=sr, mono=True)

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)), mode="constant")
    else:
        audio = audio[:target_len]

    return audio.astype(np.float32)


def extract_segments(audio, sr=SR):
    """
    Split 30-second audio into five non-overlapping 6-second segments.
    """

    segment_len = int(SEGMENT_DURATION * sr)
    segments = []

    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len

        segment = audio[start:end]

        if len(segment) == segment_len:
            segments.append(segment.astype(np.float32))

    return segments


# ============================================================
# Leakage Check
# ============================================================

def check_no_data_leakage(train_files, test_files):
    """
    Ensure no original recording appears in both train and test.
    """

    train_base = set(os.path.basename(f) for f in train_files)
    test_base = set(os.path.basename(f) for f in test_files)

    overlap = train_base.intersection(test_base)

    if len(overlap) > 0:
        raise RuntimeError(f"Data leakage detected: {list(overlap)[:10]}")

    print("\nData leakage check PASSED.")
    print("No original 30-second recording appears in both train and test.")


# ============================================================
# PaSST Embedding Extraction
# ============================================================

def extract_passt_embedding(audio_segment, passt_model, device, sr=SR):
    """
    Extract PaSST pretrained representation from one raw audio segment.

    Input:
        audio_segment: np.ndarray, shape = (audio_samples,)

    Output:
        embedding: np.ndarray
    """

    if sr != 32000:
        print("Warning: PaSST expects 32000 Hz audio.")

    audio_segment = audio_segment.astype(np.float32)

    audio_tensor = torch.tensor(audio_segment, dtype=torch.float32)

    # Shape: (batch, samples)
    if audio_tensor.dim() == 1:
        audio_tensor = audio_tensor.unsqueeze(0)

    audio_tensor = audio_tensor.to(device)

    with torch.no_grad():
        outputs = passt_model(audio_tensor)

    # Handle possible output formats
    if isinstance(outputs, (tuple, list)):
        outputs = outputs[0]

    if isinstance(outputs, dict):
        if "logits" in outputs:
            outputs = outputs["logits"]
        else:
            outputs = list(outputs.values())[0]

    if outputs.dim() > 2:
        outputs = outputs.mean(dim=1)

    embedding = outputs.detach().cpu().numpy()[0].astype(np.float32)

    return embedding


def extract_passt_embeddings_from_files(
    file_paths,
    labels,
    passt_model,
    device,
    split_name="train"
):
    """
    Extract PaSST embeddings for all 6-second segments.
    """

    X_embeddings = []
    y_out = []
    metadata = []

    for file_idx, (file_path, label) in enumerate(zip(file_paths, labels)):
        print(f"[{split_name}] {file_idx + 1}/{len(file_paths)}: {os.path.basename(file_path)}")

        try:
            audio = load_audio_30s(file_path)
            segments = extract_segments(audio)

            for segment_id, segment in enumerate(segments):
                embedding = extract_passt_embedding(
                    audio_segment=segment,
                    passt_model=passt_model,
                    device=device,
                    sr=SR
                )

                X_embeddings.append(embedding)
                y_out.append(int(label))

                metadata.append({
                    "split": split_name,
                    "file_path": file_path,
                    "file_name": os.path.basename(file_path),
                    "segment_id": segment_id,
                    "label": int(label),
                    "genre": GENRES[int(label)]
                })

        except Exception as e:
            print(f"Skipping file due to error: {file_path}")
            print("Error:", e)

    return (
        np.array(X_embeddings, dtype=np.float32),
        np.array(y_out, dtype=np.int64),
        metadata
    )


# ============================================================
# Save Metadata
# ============================================================

def save_metadata(metadata, save_path):
    """
    Save metadata for reproducibility.
    """

    if len(metadata) == 0:
        return

    keys = metadata[0].keys()

    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(metadata)


# ============================================================
# Train SVM Classifier
# ============================================================

def train_passt_svm_classifier(X_train, y_train, X_test, y_test):
    """
    Train SVM classifier on official PaSST embeddings.
    """

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    classifier = SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        probability=True,
        random_state=RANDOM_STATE
    )

    print("\nTraining SVM classifier on official PaSST embeddings...")
    classifier.fit(X_train_scaled, y_train)

    y_pred = classifier.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    print("\n" + "=" * 70)
    print("Official Pretrained PaSST + SVM Results")
    print("=" * 70)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=GENRES, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    result = {
        "Model": "Official_PaSST_Pretrained",
        "Input": "Raw waveform",
        "Feature": "PaSST pretrained AudioSet logits",
        "Classifier": "SVM-RBF",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

    return classifier, scaler, y_pred, result


# ============================================================
# Run Official PaSST Experiment
# ============================================================

def run_official_passt_experiment():
    """
    Complete PaSST experiment:
        1. Collect original WAV files.
        2. Split original files before segmentation.
        3. Segment inside train/test only.
        4. Extract PaSST embeddings.
        5. Train SVM classifier.
        6. Save results.
    """

    print("=" * 70)
    print("Running Official Pretrained PaSST Baseline")
    print("=" * 70)

    # --------------------------------------------------------
    # Collect original files
    # --------------------------------------------------------
    file_paths, labels = collect_original_files(DATA_PATH, GENRES)

    print("\nTotal original files:", len(file_paths))

    # --------------------------------------------------------
    # Split original files before segmentation
    # --------------------------------------------------------
    train_files, test_files, y_train_files, y_test_files = train_test_split(
        file_paths,
        labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=labels
    )

    print("\nOriginal-file split:")
    print("Train original files:", len(train_files))
    print("Test original files :", len(test_files))

    check_no_data_leakage(train_files, test_files)

    # --------------------------------------------------------
    # Save original split
    # --------------------------------------------------------
    split_csv = os.path.join(META_DIR, "official_passt_original_split.csv")

    with open(split_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["split", "file_path", "label", "genre"])

        for fp, lb in zip(train_files, y_train_files):
            writer.writerow(["train", fp, int(lb), GENRES[int(lb)]])

        for fp, lb in zip(test_files, y_test_files):
            writer.writerow(["test", fp, int(lb), GENRES[int(lb)]])

    print("Original split saved to:", split_csv)

    # --------------------------------------------------------
    # Load official PaSST
    # --------------------------------------------------------
    passt_model, device = create_official_passt_model()

    # --------------------------------------------------------
    # Embedding file paths
    # --------------------------------------------------------
    train_emb_path = os.path.join(EMB_DIR, "X_train_passt_official.npy")
    train_y_path = os.path.join(EMB_DIR, "y_train_passt_official.npy")
    test_emb_path = os.path.join(EMB_DIR, "X_test_passt_official.npy")
    test_y_path = os.path.join(EMB_DIR, "y_test_passt_official.npy")

    # --------------------------------------------------------
    # Load existing embeddings or extract new ones
    # --------------------------------------------------------
    if (
        os.path.exists(train_emb_path)
        and os.path.exists(train_y_path)
        and os.path.exists(test_emb_path)
        and os.path.exists(test_y_path)
    ):
        print("\nLoading saved PaSST embeddings...")

        X_train_passt = np.load(train_emb_path)
        y_train_passt = np.load(train_y_path)
        X_test_passt = np.load(test_emb_path)
        y_test_passt = np.load(test_y_path)

    else:
        print("\nExtracting PaSST embeddings for training split...")
        X_train_passt, y_train_passt, train_metadata = extract_passt_embeddings_from_files(
            train_files,
            y_train_files,
            passt_model,
            device,
            split_name="train"
        )

        print("\nExtracting PaSST embeddings for testing split...")
        X_test_passt, y_test_passt, test_metadata = extract_passt_embeddings_from_files(
            test_files,
            y_test_files,
            passt_model,
            device,
            split_name="test"
        )

        np.save(train_emb_path, X_train_passt)
        np.save(train_y_path, y_train_passt)
        np.save(test_emb_path, X_test_passt)
        np.save(test_y_path, y_test_passt)

        save_metadata(
            train_metadata,
            os.path.join(META_DIR, "official_passt_train_metadata.csv")
        )

        save_metadata(
            test_metadata,
            os.path.join(META_DIR, "official_passt_test_metadata.csv")
        )

        print("\nPaSST embeddings saved to:", EMB_DIR)

    print("\nEmbedding shapes:")
    print("X_train_passt:", X_train_passt.shape)
    print("y_train_passt:", y_train_passt.shape)
    print("X_test_passt :", X_test_passt.shape)
    print("y_test_passt :", y_test_passt.shape)

    # --------------------------------------------------------
    # Train classifier and evaluate
    # --------------------------------------------------------
    classifier, scaler, y_pred, result = train_passt_svm_classifier(
        X_train_passt,
        y_train_passt,
        X_test_passt,
        y_test_passt
    )

    # --------------------------------------------------------
    # Save final result
    # --------------------------------------------------------
    result_csv = os.path.join(SAVE_DIR, "official_passt_result.csv")
    pd.DataFrame([result]).to_csv(result_csv, index=False)

    print("\nFinal PaSST result saved to:", result_csv)

    return {
        "classifier": classifier,
        "scaler": scaler,
        "y_pred": y_pred,
        "result": result,
        "X_train_passt": X_train_passt,
        "y_train_passt": y_train_passt,
        "X_test_passt": X_test_passt,
        "y_test_passt": y_test_passt
    }


# ============================================================
# Execute
# ============================================================

passt_outputs = run_official_passt_experiment()

print("\nOfficial PaSST comparison row:")
print(passt_outputs["result"])

Running Official Pretrained PaSST Baseline
adeni: 230 original WAV files
hadrami: 230 original WAV files
lahji: 230 original WAV files
sanaai: 230 original WAV files
tihami: 230 original WAV files

Total original files: 1150

Original-file split:
Train original files: 920
Test original files : 230

Data leakage check PASSED.
No original 30-second recording appears in both train and test.
Original split saved to: /root/autodl-tmp/official_passt_results/metadata/official_passt_original_split.csv


 Loading PASST TRAINED ON AUDISET 




100%|██████████| 329M/329M [00:35<00:00, 9.77MB/s] 
/root/miniconda3/lib/python3.8/site-packages/hear21passt/models/helpers/vit_helpers.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experime

PaSST(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
     

/root/miniconda3/lib/python3.8/site-packages/torch/functional.py:666: UserWarning: stft with return_complex=False is deprecated. In a future pytorch release, stft will return complex tensors for all inputs, and return_complex=False will raise an error.
Note: you can still call torch.view_as_real on the complex output to recover the old return format. (Triggered internally at ../aten/src/ATen/native/SpectralOps.cpp:873.)
  return _VF.stft(input, n_fft, hop_length, win_length, window,  # type: ignore[attr-defined]
/root/miniconda3/lib/python3.8/site-packages/hear21passt/models/preprocess.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/root/miniconda3/lib/python3.8/site-packages/hear21passt/models/passt.py:304: UserWarning: Input image size (128*600) doesn't match model (128*998).
  warnings.warn(f"Input image size ({H}*{W}) doesn't match model ({self.img_size[

x torch.Size([1, 1, 128, 600])
self.norm(x) torch.Size([1, 768, 12, 59])
 patch_embed :  torch.Size([1, 768, 12, 59])
 self.time_new_pos_embed.shape torch.Size([1, 768, 1, 99])
 CUT time_new_pos_embed.shape torch.Size([1, 768, 1, 59])
 self.freq_new_pos_embed.shape torch.Size([1, 768, 12, 1])
X flattened torch.Size([1, 708, 768])
 self.new_pos_embed.shape torch.Size([1, 2, 768])
 self.cls_tokens.shape torch.Size([1, 1, 768])
 self.dist_token.shape torch.Size([1, 1, 768])
 final sequence x torch.Size([1, 710, 768])
 after 12 atten blocks x torch.Size([1, 710, 768])
forward_features torch.Size([1, 768])
head torch.Size([1, 527])
[train] 2/920: 0889_0001_162_47_3.wav
[train] 3/920: 0175_0002_214_63_5.wav
[train] 4/920: 0832_0003_124_46_2.wav
[train] 5/920: 0928_0001_167_50_3.wav
[train] 6/920: 0102_0010_195_66_5.wav
[train] 7/920: 0290_0004_022_11_4.wav
[train] 8/920: 0478_0002_110_30_2.wav
[train] 9/920: 0287_0001_022_11_4.wav
[train] 10/920: 0863_0004_061_24_1.wav
[train] 11/920: 0323_0